In [1]:

import pandas as pd
import numpy as np

from sklearn.model_selection import KFold, RandomizedSearchCV
from sklearn.base import clone
from sklearn.compose import TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    median_absolute_error,
    mean_absolute_percentage_error
)

Đọc dữ liệu

In [2]:

df = pd.read_csv("zillow_final.csv")
df.columns = df.columns.str.replace("\ufeff", "", regex=False).str.strip()

print("Shape:", df.shape)
df.head()

Shape: (4251, 32)


,price,bed,bath,living,lot_sqft,lot_living,bed_bath,type_condo,type_manufactured,type_multi,...,risk_overall,risk_loss,risk_social,risk_resilience,risk_fire,risk_earthquake,risk_heat,dist_city,dist_airport,dist_coast
0,4980000.0,4.0,5.0,4126.0,4922.000,1.192923,0.8,False,False,False,...,80.718966,89.710407,9.209856,12.692809,73.920540,94.138632,8.354783,13.715619,17.888756,13.623171
1,1215000.0,3.0,2.0,1825.0,7840.800,4.296329,1.5,False,False,False,...,75.714982,83.604998,23.062252,12.692809,18.467649,89.615069,13.904381,39.379252,34.192540,22.666476
2,2629000.0,4.0,4.0,3019.0,43381.404,14.369461,1.0,False,False,False,...,92.322785,96.534514,9.209856,12.692809,98.160370,93.614213,13.158396,18.768005,19.185657,11.646300
3,400000.0,3.0,2.0,944.0,4268.880,4.522119,1.5,False,False,True,...,68.423055,43.952134,93.082983,12.692809,0.000000,93.975717,20.717146,9.473542,11.193127,18.586372
4,849000.0,2.0,2.0,1154.0,5002.000,4.334489,1.0,False,False,False,...,53.028195,54.870000,40.578096,12.692809,0.000000,87.909814,8.907717,37.911992,27.545766,35.124601


Tách X và y với toàn bộ feature

In [3]:

df = df.dropna(subset=["price"])

feature_cols = [col for col in df.columns if col != "price"]

X = df[feature_cols].copy()
y = df["price"]

# Đổi True/False thành 0/1 nếu có
for col in X.select_dtypes(include=["bool"]).columns:
    X[col] = X[col].astype(int)

# Ép toàn bộ feature về numeric và điền missing value nếu có
X = X.apply(pd.to_numeric, errors="coerce")
X = X.fillna(X.median())

print("Feature count:", len(feature_cols))
print("Features:", feature_cols)
print("X shape:", X.shape)
print("y shape:", y.shape)

Feature count: 31
Features: ['bed', 'bath', 'living', 'lot_sqft', 'lot_living', 'bed_bath', 'type_condo', 'type_manufactured', 'type_multi', 'type_single', 'type_townhouse', 'lat', 'long', 'income', 'poverty', 'unemployment', 'pop', 'bachelor', 'rent', 'owner_occ', 'pop_density', 'risk_overall', 'risk_loss', 'risk_social', 'risk_resilience', 'risk_fire', 'risk_earthquake', 'risk_heat', 'dist_city', 'dist_airport', 'dist_coast']
X shape: (4251, 31)
y shape: (4251,)


Tách 10 căn sample và nhập tay lại

In [4]:

# 10 căn này được lấy từ sample cố định, sau đó nhập tay lại ở manual_houses
prediction_index = [1024, 2179, 4104, 3622, 4130, 721, 3518, 1113, 1138, 2530]

# Loại 10 căn sample ra khỏi dữ liệu train/CV để so sánh dự đoán công bằng hơn
X_model = X.drop(index=prediction_index).copy()
y_model = y.drop(index=prediction_index).copy()

actual_prices = y.loc[prediction_index].copy()

manual_houses = pd.DataFrame([
    {"bed": 3.0, "bath": 2.0, "living": 1912.0, "lot_sqft": 8712.0, "lot_living": 4.556485355648536, "bed_bath": 1.5, "type_condo": 0, "type_manufactured": 0, "type_multi": 0, "type_single": 1, "type_townhouse": 0, "lat": 35.32581, "long": -119.072464, "income": 56131.0, "poverty": 0.2940742108177958, "unemployment": 0.0983326207781103, "pop": 5456.0, "bachelor": 0.1423335369578497, "rent": 1569.0, "owner_occ": 0.3506925207756233, "pop_density": 3787.234603773768, "risk_overall": 38.617958688594776, "risk_loss": 46.58148765765554, "risk_social": 24.50986199, "risk_resilience": 2.9404099040398055, "risk_fire": 0.0, "risk_earthquake": 84.12947569952316, "risk_heat": 90.13060679727346, "dist_city": 160.6148419332613, "dist_airport": 165.46996236938077, "dist_coast": 115.69487221775992},
    {"bed": 1.0, "bath": 1.0, "living": 924.0, "lot_sqft": 9239.076, "lot_living": 9.998999999999999, "bed_bath": 1.0, "type_condo": 1, "type_manufactured": 0, "type_multi": 0, "type_single": 0, "type_townhouse": 0, "lat": 38.488213, "long": -121.45701, "income": 24386.0, "poverty": 0.3129451193967323, "unemployment": 0.3386923901393355, "pop": 2424.0, "bachelor": 0.0502024291497975, "rent": 1221.0, "owner_occ": 0.1686121919584954, "pop_density": 7646.470604935507, "risk_overall": 1.738551365749825, "risk_loss": 0.705780116025084, "risk_social": 96.40052123, "risk_resilience": 43.58488330766497, "risk_fire": 0.0, "risk_earthquake": 75.0668902286754, "risk_heat": 26.39305972639306, "dist_city": 10.88153339554928, "dist_airport": 25.776509261355983, "dist_coast": 109.60256392983328},
    {"bed": 3.0, "bath": 2.0, "living": 1276.0, "lot_sqft": 20472.999999999996, "lot_living": 16.044670846394983, "bed_bath": 1.5, "type_condo": 0, "type_manufactured": 0, "type_multi": 0, "type_single": 1, "type_townhouse": 0, "lat": 39.74341, "long": -121.62628, "income": 94167.0, "poverty": 0.1180960933991917, "unemployment": 0.0, "pop": 2248.0, "bachelor": 0.357944477259303, "rent": 1125.0, "owner_occ": 0.8545454545454545, "pop_density": 16.428612442681736, "risk_overall": 73.74930136872273, "risk_loss": 73.73816849472719, "risk_social": 51.10466149, "risk_resilience": 48.271531809027366, "risk_fire": 98.76208483464735, "risk_earthquake": 88.31293924583497, "risk_heat": 17.175508842175507, "dist_city": 129.68690002347853, "dist_airport": 116.6069758702642, "dist_coast": 172.64631829083157},
    {"bed": 2.0, "bath": 2.0, "living": 1900.0, "lot_sqft": 260924.4, "lot_living": 137.32863157894735, "bed_bath": 1.0, "type_condo": 0, "type_manufactured": 0, "type_multi": 0, "type_single": 1, "type_townhouse": 0, "lat": 34.107998, "long": -118.897575, "income": 209750.0, "poverty": 0.0708549834671705, "unemployment": 0.0405244338498212, "pop": 2117.0, "bachelor": 0.6539624924379915, "rent": 2380.0, "owner_occ": 0.8299155609167672, "pop_density": 14.782009145545969, "risk_overall": 93.8900978678368, "risk_loss": 94.713108016065, "risk_social": 45.74720133, "risk_resilience": 27.489633929629193, "risk_fire": 99.34596220850725, "risk_earthquake": 90.73882487246264, "risk_heat": 9.254492587825922, "dist_city": 60.539171865459046, "dist_airport": 48.72207190153674, "dist_coast": 38.705306827320015},
    {"bed": 2.0, "bath": 2.0, "living": 1215.0, "lot_sqft": 13503.6, "lot_living": 11.114074074074074, "bed_bath": 1.0, "type_condo": 0, "type_manufactured": 1, "type_multi": 0, "type_single": 0, "type_townhouse": 0, "lat": 39.741573, "long": -121.58215, "income": 44671.0, "poverty": 0.3300173010380622, "unemployment": 0.0849582172701949, "pop": 2321.0, "bachelor": 0.302915082382763, "rent": 1682.0, "owner_occ": 0.7965895249695494, "pop_density": 30.710603267790784, "risk_overall": 65.64042191383349, "risk_loss": 52.87948892594593, "risk_social": 79.71272878, "risk_resilience": 48.271531809027366, "risk_fire": 98.88932491408322, "risk_earthquake": 82.49794869965396, "risk_heat": 23.000381333714667, "dist_city": 129.2049574445451, "dist_airport": 116.365034994374, "dist_coast": 174.57820090729555},
    {"bed": 3.0, "bath": 3.0, "living": 1175.0, "lot_sqft": 111213.036, "lot_living": 94.64939234042552, "bed_bath": 1.0, "type_condo": 0, "type_manufactured": 0, "type_multi": 0, "type_single": 0, "type_townhouse": 1, "lat": 37.729008, "long": -122.37143, "income": 154375.0, "poverty": 0.0656370656370656, "unemployment": 0.0340599455040872, "pop": 1295.0, "bachelor": 0.467379679144385, "rent": 2861.0, "owner_occ": 0.6620450606585788, "pop_density": 625.6165584036004, "risk_overall": 4.964741417240436, "risk_loss": 5.318599243723137, "risk_social": 45.04412723, "risk_resilience": 99.67183982940408, "risk_fire": 0.0, "risk_earthquake": 81.18511647818487, "risk_heat": 8.462033462033462, "dist_city": 6.620205477164951, "dist_airport": 11.995115941992594, "dist_coast": 6.620205477164951},
    {"bed": 4.0, "bath": 3.0, "living": 3816.0, "lot_sqft": 64468.8, "lot_living": 16.89433962264151, "bed_bath": 1.3333333333333333, "type_condo": 0, "type_manufactured": 0, "type_multi": 0, "type_single": 1, "type_townhouse": 0, "lat": 38.54327, "long": -121.79572, "income": 123347.0, "poverty": 0.110494880546075, "unemployment": 0.0231170768083519, "pop": 2344.0, "bachelor": 0.843421052631579, "rent": 2345.0, "owner_occ": 0.6236559139784946, "pop_density": 950.2397085066724, "risk_overall": 45.56740751311049, "risk_loss": 45.49404608121756, "risk_social": 47.36421252, "risk_resilience": 58.34024404691387, "risk_fire": 80.38481205332192, "risk_earthquake": 84.92740180514431, "risk_heat": 27.066352066352067, "dist_city": 26.543188547102808, "dist_airport": 24.535173792782963, "dist_coast": 80.76669179846614},
    {"bed": 4.0, "bath": 2.0, "living": 2698.0, "lot_sqft": 17859.6, "lot_living": 6.619570051890289, "bed_bath": 2.0, "type_condo": 0, "type_manufactured": 0, "type_multi": 0, "type_single": 1, "type_townhouse": 0, "lat": 35.379585, "long": -119.14456, "income": 123523.0, "poverty": 0.023649980291683, "unemployment": 0.0320723684210526, "pop": 2537.0, "bachelor": 0.4422081656124209, "rent": 1162.0, "owner_occ": 0.8403648802736602, "pop_density": 979.1726087944908, "risk_overall": 24.61203667368271, "risk_loss": 36.08873335368861, "risk_social": 9.209856068, "risk_resilience": 2.9404099040398055, "risk_fire": 0.0, "risk_earthquake": 82.394491812636, "risk_heat": 73.81429048095714, "dist_city": 161.6935053216189, "dist_airport": 173.48942839044702, "dist_coast": 117.96293485428856},
    {"bed": 4.0, "bath": 3.0, "living": 2488.0, "lot_sqft": 10454.4, "lot_living": 4.201929260450161, "bed_bath": 1.3333333333333333, "type_condo": 0, "type_manufactured": 0, "type_multi": 0, "type_single": 1, "type_townhouse": 0, "lat": 35.41577, "long": -119.14655, "income": 120343.0, "poverty": 0.0997838616714697, "unemployment": 0.1430678466076696, "pop": 2786.0, "bachelor": 0.2730819245773732, "rent": 2273.0, "owner_occ": 0.8561064087061668, "pop_density": 1573.186487536478, "risk_overall": 18.417704208435897, "risk_loss": 28.23120464100336, "risk_social": 9.209856068, "risk_resilience": 2.9404099040398055, "risk_fire": 55.687155886934704, "risk_earthquake": 82.59070315008384, "risk_heat": 78.09476142809476, "dist_city": 157.871796916336, "dist_airport": 177.26817893963488, "dist_coast": 121.5316959949303},
    {"bed": 1.0, "bath": 1.0, "living": 1063.0, "lot_sqft": 14373.999999999996, "lot_living": 13.522107243650044, "bed_bath": 1.0, "type_condo": 0, "type_manufactured": 0, "type_multi": 0, "type_single": 1, "type_townhouse": 0, "lat": 37.56571, "long": -118.75058, "income": 112721.0, "poverty": 0.1527541169789892, "unemployment": 0.0167115902964959, "pop": 3532.0, "bachelor": 0.4543775397118581, "rent": 804.0, "owner_occ": 0.8254568367989918, "pop_density": 0.8524082722028352, "risk_overall": 98.25074619766212, "risk_loss": 97.079409070625, "risk_social": 79.69732867, "risk_resilience": 54.031512853927254, "risk_fire": 98.08188553149488, "risk_earthquake": 92.00765818795855, "risk_heat": 47.38786405453072, "dist_city": 130.0513826294961, "dist_airport": 278.3443075752705, "dist_coast": 298.83598931848263}
], index=prediction_index)

# Đưa dữ liệu nhập tay về đúng thứ tự cột mà model đã học
X_predict_10 = manual_houses.reindex(columns=X.columns)

prediction_preview_cols = [
    col for col in ["bed", "bath", "living", "lot_sqft", "type_condo", "type_single", "type_townhouse", "dist_city", "dist_airport", "dist_coast"]
    if col in X_predict_10.columns
]

print("Train/CV data:", X_model.shape)
print("Manual prediction data:", X_predict_10.shape)

pd.concat([
    actual_prices.rename("Actual Price"),
    X_predict_10[prediction_preview_cols]
], axis=1)

Train/CV data: (4241, 31)
Manual prediction data: (10, 31)


,Actual Price,bed,bath,living,lot_sqft,type_condo,type_single,type_townhouse,dist_city,dist_airport,dist_coast
1024,464900.0,3.0,2.0,1912.0,8712.000,0,1,0,160.614842,165.469962,115.694872
2179,148900.0,1.0,1.0,924.0,9239.076,1,0,0,10.881533,25.776509,109.602564
4104,415000.0,3.0,2.0,1276.0,20473.000,0,1,0,129.686900,116.606976,172.646318
3622,2499000.0,2.0,2.0,1900.0,260924.400,0,1,0,60.539172,48.722072,38.705307
4130,349900.0,2.0,2.0,1215.0,13503.600,0,0,0,129.204957,116.365035,174.578201
721,688000.0,3.0,3.0,1175.0,111213.036,0,0,1,6.620205,11.995116,6.620205
3518,2950000.0,4.0,3.0,3816.0,64468.800,0,1,0,26.543189,24.535174,80.766692
1113,679000.0,4.0,2.0,2698.0,17859.600,0,1,0,161.693505,173.489428,117.962935
1138,663000.0,4.0,3.0,2488.0,10454.400,0,1,0,157.871797,177.268179,121.531696
2530,585000.0,1.0,1.0,1063.0,14374.000,0,1,0,130.051383,278.344308,298.835989


Hàm đánh giá model

In [5]:

def regression_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    median_ae = median_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = mse ** 0.5
    r2 = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100

    return {
        "MAE": mae,
        "Median AE": median_ae,
        "MSE": mse,
        "RMSE": rmse,
        "R2": r2,
        "MAPE (%)": mape
    }


def evaluate_model(model_name, y_true, y_pred):
    metrics = regression_metrics(y_true, y_pred)

    print("=" * 60)
    print(model_name)
    print("=" * 60)
    print(f"MAE       : {metrics['MAE']:,.0f}")
    print(f"Median AE : {metrics['Median AE']:,.0f}")
    print(f"MSE       : {metrics['MSE']:,.0f}")
    print(f"RMSE      : {metrics['RMSE']:,.0f}")
    print(f"R2        : {metrics['R2']:.4f}")
    print(f"MAPE      : {metrics['MAPE (%)']:.2f}%")
    print()

    return metrics


def cross_validate_model(model_name, model, X_data, y_data, cv):
    fold_results = []

    for fold, (train_idx, val_idx) in enumerate(cv.split(X_data), start=1):
        X_train_fold = X_data.iloc[train_idx]
        X_val_fold = X_data.iloc[val_idx]
        y_train_fold = y_data.iloc[train_idx]
        y_val_fold = y_data.iloc[val_idx]

        fold_model = clone(model)
        fold_model.fit(X_train_fold, np.log1p(y_train_fold))

        y_pred_fold = np.expm1(fold_model.predict(X_val_fold))
        metrics = regression_metrics(y_val_fold, y_pred_fold)
        metrics["Model"] = model_name
        metrics["Fold"] = fold
        fold_results.append(metrics)

    return pd.DataFrame(fold_results)

Khai báo model

In [6]:

models = {
    "Linear Regression thuần túy": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LinearRegression())
    ]),
    "Random Forest": RandomForestRegressor(
        n_estimators=300,
        max_depth=12,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ),
    "XGBoost": XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1,
        tree_method="hist"
    )
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)

list(models.keys())

['Linear Regression thuần túy', 'Random Forest', 'XGBoost']

Cross-validation 5-fold

In [7]:

cv_results = pd.concat(
    [
        cross_validate_model(model_name, model, X_model, y_model, kf)
        for model_name, model in models.items()
    ],
    ignore_index=True
)

cv_results

,MAE,Median AE,MSE,RMSE,R2,MAPE (%),Model,Fold
0,1.064416e+06,179439.255679,1.395841e+14,1.181457e+07,-52.684166,61.398986,Linear Regression thuần túy,1
1,6.542045e+05,202651.209827,1.141685e+13,3.378883e+06,-3.035548,34.477196,Linear Regression thuần túy,2
2,7.336686e+05,174928.210329,3.356596e+13,5.793614e+06,-11.296799,37.024971,Linear Regression thuần túy,3
3,5.219170e+05,207594.984444,1.296049e+12,1.138441e+06,0.613139,30.046411,Linear Regression thuần túy,4
4,5.152122e+05,186877.940611,1.889061e+12,1.374431e+06,0.388332,28.850929,Linear Regression thuần túy,5
5,3.366581e+05,110071.502765,5.157873e+11,7.181834e+05,0.801628,19.636840,Random Forest,1
6,3.536327e+05,117188.222692,5.747895e+11,7.581487e+05,0.796828,21.268471,Random Forest,2
7,3.309653e+05,120355.916690,4.640801e+11,6.812343e+05,0.829985,21.668541,Random Forest,3
8,3.834322e+05,115887.717595,7.393035e+11,8.598276e+05,0.779323,20.528819,Random Forest,4
9,3.349888e+05,111510.690200,5.760913e+11,7.590068e+05,0.813465,18.492356,Random Forest,5


Bảng kết quả cross-validation

In [8]:
cv_summary = (
    cv_results
    .groupby("Model")
    .agg(
        MAE_mean=("MAE", "mean"),
        MAE_std=("MAE", "std"),
        Median_AE_mean=("Median AE", "mean"),
        MSE_mean=("MSE", "mean"),
        MSE_std=("MSE", "std"),
        RMSE_mean=("RMSE", "mean"),
        RMSE_std=("RMSE", "std"),
        R2_mean=("R2", "mean"),
        R2_std=("R2", "std"),
        MAPE_mean=("MAPE (%)", "mean"),
        MAPE_std=("MAPE (%)", "std")
    )
    .sort_values("RMSE_mean")
    .reset_index()
)

best_model_name = cv_summary.iloc[0]["Model"]

cv_summary

,Model,MAE_mean,MAE_std,Median_AE_mean,MSE_mean,MSE_std,RMSE_mean,RMSE_std,R2_mean,R2_std,MAPE_mean,MAPE_std
0,XGBoost,331574.059850,20296.593947,114822.187500,5.056621e+11,9.248511e+10,7.088124e+05,6.370903e+04,0.827472,0.018552,19.620532,1.195774
1,Random Forest,347935.422837,21652.131656,115002.809989,5.740104e+11,1.034424e+11,7.552802e+05,6.672940e+04,0.804246,0.018911,20.319005,1.281479
2,Linear Regression thuần túy,697883.732882,224648.523690,190298.320178,3.755040e+13,5.851456e+13,4.699988e+06,4.396093e+06,-13.203008,22.590834,38.359699,13.296536


RandomizedSearchCV cho model tốt nhất

In [9]:
random_search_spaces = {
    "Random Forest": {
        "regressor__n_estimators": [200, 300, 500],
        "regressor__max_depth": [8, 12, 16, None],
        "regressor__min_samples_split": [2, 5, 10],
        "regressor__min_samples_leaf": [1, 2, 4],
        "regressor__max_features": ["sqrt", "log2", 1.0]
    },
    "XGBoost": {
        "regressor__n_estimators": [200, 300, 500],
        "regressor__learning_rate": [0.03, 0.05, 0.08, 0.1],
        "regressor__max_depth": [3, 4, 5, 6],
        "regressor__subsample": [0.75, 0.9, 1.0],
        "regressor__colsample_bytree": [0.75, 0.9, 1.0],
        "regressor__reg_lambda": [0.5, 1.0, 2.0]
    }
}

if best_model_name in random_search_spaces:
    search_estimator = TransformedTargetRegressor(
        regressor=clone(models[best_model_name]),
        func=np.log1p,
        inverse_func=np.expm1,
        check_inverse=False
    )

    random_search = RandomizedSearchCV(
        estimator=search_estimator,
        param_distributions=random_search_spaces[best_model_name],
        n_iter=12,
        scoring="neg_root_mean_squared_error",
        cv=kf,
        random_state=42,
        n_jobs=-1,
        verbose=0
    )

    random_search.fit(X_model, y_model)
    tuned_best_model = random_search.best_estimator_

    random_search_results = pd.DataFrame(random_search.cv_results_)[[
        "mean_test_score",
        "std_test_score",
        "rank_test_score",
        "params"
    ]].copy()
    random_search_results["RMSE_mean"] = -random_search_results["mean_test_score"]
    random_search_results["RMSE_std"] = random_search_results["std_test_score"]
    random_search_results = random_search_results.sort_values("rank_test_score")

    print("Best model before tuning:", best_model_name)
    print("Best RandomizedSearchCV RMSE:", -random_search.best_score_)
    print("Best params:", random_search.best_params_)
else:
    random_search = None
    tuned_best_model = None
    random_search_results = pd.DataFrame()
    print(f"Không tune {best_model_name} vì chưa có search space phù hợp.")

random_search_results[["RMSE_mean", "RMSE_std", "rank_test_score", "params"]].head()

Best model before tuning: XGBoost
Best RandomizedSearchCV RMSE: 687113.0521441939
Best params: {'regressor__subsample': 0.9, 'regressor__reg_lambda': 0.5, 'regressor__n_estimators': 500, 'regressor__max_depth': 4, 'regressor__learning_rate': 0.08, 'regressor__colsample_bytree': 1.0}


,RMSE_mean,RMSE_std,rank_test_score,params
0,687113.052144,52490.416195,1,"{'regressor__subsample': 0.9, 'regressor__reg_..."
5,688613.138206,43628.120427,2,"{'regressor__subsample': 0.75, 'regressor__reg..."
2,689859.568189,47636.380634,3,"{'regressor__subsample': 0.9, 'regressor__reg_..."
8,691018.051069,54908.458723,4,"{'regressor__subsample': 1.0, 'regressor__reg_..."
3,691684.400091,61103.646536,5,"{'regressor__subsample': 1.0, 'regressor__reg_..."


Nhận xét model tốt hay chưa tốt

Ba model được đánh giá trên toàn bộ 31 feature trong `zillow_final.csv` trừ `price` bằng 5-fold cross-validation. Kết quả cho thấy **XGBoost** tốt nhất với RMSE trung bình khoảng **708,812**, R² trung bình khoảng **0.827** và MAPE khoảng **19.62%**. **Random Forest** đứng thứ hai với RMSE khoảng **755,280**, R² khoảng **0.804** và MAPE khoảng **20.32%**. **Linear Regression thuần túy** vẫn kém hơn đáng kể do quan hệ giữa feature và giá nhà có tính phi tuyến mạnh. Sau RandomizedSearchCV, XGBoost được cải thiện thêm với RMSE tốt nhất khoảng **687,113**, nên model cuối cùng nên ưu tiên dùng **tuned XGBoost**.

Train model cuối cùng

In [10]:
y_model_log = np.log1p(y_model)

final_models = {}

for model_name, model in models.items():
    final_model = clone(model)
    final_model.fit(X_model, y_model_log)
    final_models[model_name] = final_model

best_model_name = cv_summary.iloc[0]["Model"]
best_model = final_models[best_model_name]

linear_model = final_models["Linear Regression thuần túy"]

print(f"Best model theo RMSE trung bình: {best_model_name}")
print("Final train size:", X_model.shape)

Best model theo RMSE trung bình: XGBoost
Final train size: (4241, 31)


Dự đoán 10 căn nhập tay và so sánh giá thật

In [11]:
prediction_results = pd.concat([
    actual_prices.rename("Actual Price"),
    X_predict_10[prediction_preview_cols]
], axis=1)

prediction_results.insert(0, "Sample Index", prediction_results.index)

for model_name, model in final_models.items():
    y_pred_10 = np.expm1(model.predict(X_predict_10))
    prediction_results[f"Predicted - {model_name}"] = y_pred_10
    prediction_results[f"Abs Error - {model_name}"] = (
        prediction_results["Actual Price"] - y_pred_10
    ).abs()

if tuned_best_model is not None:
    tuned_pred_10 = tuned_best_model.predict(X_predict_10)
    prediction_results[f"Predicted - Tuned {best_model_name}"] = tuned_pred_10
    prediction_results[f"Abs Error - Tuned {best_model_name}"] = (
        prediction_results["Actual Price"] - tuned_pred_10
    ).abs()

prediction_results

,Sample Index,Actual Price,bed,bath,living,lot_sqft,type_condo,type_single,type_townhouse,dist_city,dist_airport,dist_coast,Predicted - Linear Regression thuần túy,Abs Error - Linear Regression thuần túy,Predicted - Random Forest,Abs Error - Random Forest,Predicted - XGBoost,Abs Error - XGBoost,Predicted - Tuned XGBoost,Abs Error - Tuned XGBoost
1024,1024,464900.0,3.0,2.0,1912.0,8712.000,0,1,0,160.614842,165.469962,115.694872,4.094238e+05,5.547618e+04,4.029275e+05,6.197246e+04,3.851928e+05,7.970725e+04,3.840000e+05,80900.031250
2179,2179,148900.0,1.0,1.0,924.0,9239.076,1,0,0,10.881533,25.776509,109.602564,2.078125e+05,5.891246e+04,2.215709e+05,7.267087e+04,1.855272e+05,3.662723e+04,1.970356e+05,48135.625000
4104,4104,415000.0,3.0,2.0,1276.0,20473.000,0,1,0,129.686900,116.606976,172.646318,5.814216e+05,1.664216e+05,3.986609e+05,1.633909e+04,4.419262e+05,2.692619e+04,4.294957e+05,14495.687500
3622,3622,2499000.0,2.0,2.0,1900.0,260924.400,0,1,0,60.539172,48.722072,38.705307,2.913475e+06,4.144746e+05,3.721739e+06,1.222739e+06,3.134681e+06,6.356810e+05,3.493746e+06,994746.250000
4130,4130,349900.0,2.0,2.0,1215.0,13503.600,0,0,0,129.204957,116.365035,174.578201,1.570645e+05,1.928355e+05,1.965530e+05,1.533470e+05,1.781453e+05,1.717547e+05,1.986801e+05,151219.921875
721,721,688000.0,3.0,3.0,1175.0,111213.036,0,0,1,6.620205,11.995116,6.620205,8.658721e+05,1.778721e+05,6.529764e+05,3.502356e+04,6.176832e+05,7.031681e+04,6.087608e+05,79239.187500
3518,3518,2950000.0,4.0,3.0,3816.0,64468.800,0,1,0,26.543189,24.535174,80.766692,1.572408e+06,1.377592e+06,2.336748e+06,6.132522e+05,1.936172e+06,1.013828e+06,2.477763e+06,472237.250000
1113,1113,679000.0,4.0,2.0,2698.0,17859.600,0,1,0,161.693505,173.489428,117.962935,5.200224e+05,1.589776e+05,6.647047e+05,1.429528e+04,5.871276e+05,9.187244e+04,6.088206e+05,70179.375000
1138,1138,663000.0,4.0,3.0,2488.0,10454.400,0,1,0,157.871797,177.268179,121.531696,4.583001e+05,2.046999e+05,5.947426e+05,6.825739e+04,5.338174e+05,1.291826e+05,5.334937e+05,129506.312500
2530,2530,585000.0,1.0,1.0,1063.0,14374.000,0,1,0,130.051383,278.344308,298.835989,5.855559e+05,5.559486e+02,5.658351e+05,1.916492e+04,5.489676e+05,3.603244e+04,5.089554e+05,76044.593750


Linear Regression Coefficients

In [12]:
coef_df = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": linear_model.named_steps["model"].coef_
})

coef_df["Abs_Coefficient"] = coef_df["Coefficient"].abs()

coef_df = coef_df.sort_values(
    by="Abs_Coefficient",
    ascending=False
)

coef_df.head(15)

,Feature,Coefficient,Abs_Coefficient
11,lat,-0.951703,0.951703
12,long,-0.632819,0.632819
30,dist_coast,0.437778,0.437778
27,risk_heat,-0.287038,0.287038
29,dist_airport,-0.259073,0.259073
9,type_single,0.253278,0.253278
2,living,0.245288,0.245288
13,income,0.207968,0.207968
21,risk_overall,0.178133,0.178133
28,dist_city,0.167232,0.167232
